In [ ]:
import json
from pathlib import Path
import re
import numpy as np
import pandas as pd
# mriqc_dir = Path("/data/backed_up/shared/ME_7T_Pilot/MRIQC")  # Path to the MRIQC result folder
fmriprep_dir = Path("/data/backed_up/shared/ME_7T_Pilot/fmriprep")  # Enter the absolute path to the actual fmriprep directory
# Search for confounds TSV files in each subject's func folder
tsv_files = sorted(fmriprep_dir.glob("sub-*/func/*desc-confounds_timeseries.tsv"))
records = []
print(f"Number of confounds TSV files found: {len(tsv_files)}")

for tsv_path in tsv_files:
    fname = tsv_path.name

    # Extract sub, task, and run IDs using regex
    sub_match = re.search(r"sub-([a-zA-Z0-9]+)", fname)
    task_match = re.search(r"task-([a-zA-Z0-9]+)", fname)
    run_match = re.search(r"run-([a-zA-Z0-9]+)", fname)

    sub_id = sub_match.group(1) if sub_match else "unknown"
    task_id = task_match.group(1) if task_match else "unknown"
    run_id = run_match.group(1) if run_match else "run-01"

    # Read TSV file
    df_conf = pd.read_csv(tsv_path, sep="\t")

    if "framewise_displacement" not in df_conf.columns:
        continue

    # Pure FD time series excluding the first TR (NaN)
    fd = df_conf["framewise_displacement"].dropna().to_numpy()
    total_vols = len(fd)

    if total_vols == 0:
        continue

    mean_fd = float(np.mean(fd))

    # Calculate using the 0.4 mm and 0.5 mm thresholds
    count_04 = int(np.sum(fd > 0.4))
    perc_04 = (count_04 / total_vols) * 100

    count_05 = int(np.sum(fd > 0.5))
    perc_05 = (count_05 / total_vols) * 100

    records.append(
        {
            "Subject": f"sub-{sub_id}",
            "Task": task_id,
            "Run": f"run-{run_id}",
            "Total_Vols": total_vols,
            "Mean_FD": round(mean_fd, 3),
            "FD>0.4 (count)": count_04,
            "FD>0.4 (%)": round(perc_04, 2),
            "FD>0.5 (count)": count_05,
            "FD>0.5 (%)": round(perc_05, 2),
            "Exclude_0.4 (>20%)": perc_04 > 20.0,
            "Exclude_0.5 (>20%)": perc_05 > 20.0,
        }
    )

if records:
    df_result = pd.DataFrame(records)
    output_csv = "motion_exclusion_summary_fd05.csv"
    df_result.to_csv(output_csv, index=False)
    print(f"\nSummary complete! Total {len(df_result)} runs saved to {output_csv}\n")

    # Print the top 10 runs with the highest motion rate based on the 0.5 mm criterion
    print("[Top 10 runs by FD > 0.5 mm ratio]")
    print(
        df_result.sort_values(by="FD>0.5 (%)", ascending=False)[
            [
                "Subject",
                "Run",
                "Mean_FD",
                "FD>0.4 (%)",
                "FD>0.5 (%)",
                "Exclude_0.5 (>20%)",
            ]
        ].head(10)
    )
else:
    print("No matching confounds files were found. Please check the path.")